# 🐜 ANT — GPU Training

**937K parameter byte-level transformer** where ALL knowledge lives in persistent trie memory.

## 3-Phase Curriculum
1. **Phase A** — Base LM (no memory, all weights trainable)
2. **Phase B** — Memory Training (frozen base, train AddrNet/V_proj/tags)
3. **Phase C** — End-to-End (trie READ+WRITE every forward pass)

## GPU Optimizations
- **BF16 mixed precision** (Tensor Cores)
- **torch.compile** (inductor kernel fusion)
- **Large batches** with gradient accumulation
- **Checkpoints auto-upload to HuggingFace Hub**

## 1. Setup

In [ ]:
# Verify GPU and install dependencies
!nvidia-smi
!pip install -q datasets numpy torch huggingface_hub

import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB')
    print(f'BF16: {torch.cuda.is_bf16_supported()}')

In [ ]:
# Clone repo and login to HuggingFace
import os

REPO_DIR = '/content/ANT'
if not os.path.exists(REPO_DIR):
    !git clone https://github.com/kaaninel/ANT.git {REPO_DIR}
else:
    !cd {REPO_DIR} && git pull
os.chdir(REPO_DIR)

# HuggingFace login (set HF_TOKEN in Colab secrets)
from huggingface_hub import login, create_repo
try:
    from google.colab import userdata
    login(token=userdata.get('HF_TOKEN'))
except Exception:
    login()

HF_REPO = 'kaaninel/ANT'
try:
    create_repo(HF_REPO, exist_ok=True, repo_type='model')
    print(f'\u2713 HF repo: https://huggingface.co/{HF_REPO}')
except: pass

## 2. Configuration

Edit these values and re-run to adjust.

In [ ]:
# ================================================================
# TRAINING CONFIG
# ================================================================

# Steps per phase
STEPS_A = 3000    # Phase A: Base LM only (no memory)
STEPS_B = 2000    # Phase B: Frozen base, train AddrNet/V_proj/tags
STEPS_C = 5000    # Phase C: End-to-end with trie READ+WRITE

# Batch size
BATCH_SIZE = 64
GRAD_ACCUM = 2    # 64 x 2 = 128 effective batch

# Data sizes
N_WIKI  = 50_000
N_SHELL = 5_000
N_QA    = 20_000
N_CHAT  = 5_000

# Architecture
WINDOW_SIZE = 8
NUM_PASSES  = 4
EVAL_INTERVAL = 250
LR = 3e-4

# A100/GPU optimizations
import torch
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True
torch.backends.cudnn.benchmark = True
torch.set_float32_matmul_precision('high')

total = STEPS_A + STEPS_B + STEPS_C
eff_batch = BATCH_SIZE * GRAD_ACCUM
print(f'Total steps: {total:,}')
print(f'Batch: {BATCH_SIZE} x {GRAD_ACCUM} accum = {eff_batch} effective')
print(f'Window: W={WINDOW_SIZE} passes={NUM_PASSES}')
print(f'Data: wiki={N_WIKI:,} shell={N_SHELL:,} qa={N_QA:,} chat={N_CHAT:,}')

## 3. Load Checkpoint (Optional)

If you have a previous checkpoint on HuggingFace, load it to continue training.

In [ ]:
import os
CKPT_DIR = '/content/ant_checkpoints/train'
os.makedirs(CKPT_DIR, exist_ok=True)

RESUME_FROM = None
try:
    from huggingface_hub import hf_hub_download
    for fname in ['checkpoint_best.pt', 'checkpoint_latest.pt']:
        try:
            path = hf_hub_download(
                repo_id='kaaninel/ANT',
                filename=f'checkpoints/train/{fname}',
                local_dir='/content/ant_hf_cache'
            )
            import shutil
            dest = os.path.join(CKPT_DIR, fname)
            shutil.copy2(path, dest)
            RESUME_FROM = dest
            print(f'Downloaded {fname} from HF')
            break
        except Exception as e:
            print(f'{fname}: {e}')
except ImportError:
    pass

if RESUME_FROM:
    print(f'Will resume from: {RESUME_FROM}')
else:
    print('Starting from scratch (no checkpoint found)')

## 4. Train

All 3 phases run automatically. Checkpoints upload to HuggingFace periodically.

In [ ]:
import torch
import os

from config import ModelConfig
from model import ANT
from train import train, log_model_summary, VOCAB_SIZE

device = 'cuda'

# Build model
cfg = ModelConfig()
model = ANT(cfg, use_checkpoint=False).to(device)
log_model_summary(model, cfg)

# Load checkpoint if available
if RESUME_FROM and os.path.exists(RESUME_FROM):
    ckpt = torch.load(RESUME_FROM, map_location=device, weights_only=False)
    if 'model' in ckpt:
        state = ckpt['model']
        if any(k.startswith('_orig_mod.') for k in state):
            state = {k.replace('_orig_mod.', '', 1): v for k, v in state.items()}
        model.load_state_dict(state, strict=False)
        print(f'Loaded step={ckpt.get("step", 0)}, QA={ckpt.get("qa_accuracy", 0):.1%}')

# Train all phases
best_acc = train(
    model, cfg, device,
    output_dir=CKPT_DIR,
    steps_a=STEPS_A,
    steps_b=STEPS_B,
    steps_c=STEPS_C,
    lr=LR,
    batch_size=BATCH_SIZE,
    grad_accum=GRAD_ACCUM,
    eval_interval=EVAL_INTERVAL,
    window_size=WINDOW_SIZE,
    num_passes=NUM_PASSES,
    n_wiki=N_WIKI,
    n_shell=N_SHELL,
    n_qa=N_QA,
    n_chat=N_CHAT,
    use_bf16=True,
    use_compile=True,
    use_hf_chat=True,
    hf_repo='kaaninel/ANT',
    hf_upload_interval=2000,
)

print(f'\nTraining complete! Best QA: {best_acc:.1%}')

## 5. Test Generation

Quick test of text generation and memory-based QA.

In [ ]:
import torch
import os
from config import ModelConfig, MemoryConfig
from model import ANT
from memory import MemorySystem
from data import VOCAB_SIZE, BOS_ID, tokenize, detokenize, generate_dataset
from train import (
    generate, sliding_window_encode,
    encode_frozen, trie_write, trie_read,
)

# Load best checkpoint
best = os.path.join(CKPT_DIR, 'checkpoint_best.pt')
if not os.path.exists(best):
    best = os.path.join(CKPT_DIR, 'checkpoint_latest.pt')

ckpt = torch.load(best, map_location='cuda', weights_only=False)
cfg = ModelConfig()
model = ANT(cfg, use_checkpoint=False).cuda()
state = ckpt['model']
if any(k.startswith('_orig_mod.') for k in state):
    state = {k.replace('_orig_mod.', '', 1): v for k, v in state.items()}
model.load_state_dict(state, strict=False)
model.eval()

W = ckpt.get('window_size', 8)
P = ckpt.get('num_passes', 4)
print(f'Loaded step={ckpt["step"]}, QA={ckpt.get("qa_accuracy", 0):.1%}, W={W}, P={P}')

# Setup trie memory
mem_cfg = MemoryConfig()
mem_cfg.data_path = os.path.join(CKPT_DIR, 'memory')
memory = MemorySystem(mem_cfg)
print(f'Trie: {memory.total_entries()} entries, {memory.total_nodes()} nodes')

# --- Text generation (no memory) ---
print('\n=== Text Generation ===')
for p in ['The ', 'In the year ', '$ ls -la ']:
    ids = [BOS_ID] + tokenize(p)
    out = generate(model, ids, max_new=100, temperature=0.7, top_k=30,
                   window_size=W, num_passes=P)
    print(f'\n--- {repr(p)} ---')
    print(detokenize(out)[:200])

# --- Memory QA ---
print('\n=== Memory QA ===')
examples = generate_dataset(5, seed=99, tagged=False)
for ex in examples:
    p_ids = tokenize(ex.passage)
    p_tensor = torch.tensor([p_ids], dtype=torch.long, device='cuda')
    with torch.no_grad():
        p_hidden = sliding_window_encode(model, p_tensor, W, P, causal=True)
        trie_write(model, memory, p_hidden, temperature=0.01)
        query_ids = tokenize(ex.question)
        q_tensor = torch.tensor([query_ids], dtype=torch.long, device='cuda')
        q_hidden = sliding_window_encode(model, q_tensor, W, P, causal=True)
        mk, mv, mm = trie_read(model, memory, q_hidden[:, -1, :], 'cuda')
    prompt = [BOS_ID] + tokenize(ex.question + ' ')
    out = generate(model, prompt, max_new=50, temperature=0.3, top_k=10,
                   window_size=W, num_passes=P,
                   mem_keys=mk, mem_vals=mv, mem_mask=mm)
    answer = detokenize(out[len(prompt):])
    print(f'Q: {ex.question}')
    print(f'A: {answer[:100]}  (expected: {ex.answer})')
    print()

## 6. Upload Final Checkpoint

In [ ]:
from huggingface_hub import HfApi
import os
api = HfApi()

for f in ['checkpoint_latest.pt', 'checkpoint_best.pt']:
    path = os.path.join(CKPT_DIR, f)
    if os.path.exists(path):
        api.upload_file(
            path_or_fileobj=path,
            path_in_repo=f'checkpoints/train/{f}',
            repo_id='kaaninel/ANT', repo_type='model')
        print(f'Uploaded {f}')

print('\nDone! Check https://huggingface.co/kaaninel/ANT')